In [1]:
import pandas as pd
import os

base_path = "../data/processed/text"

resume_df = pd.read_csv(os.path.join(base_path, "unified_resumes.csv"))
jd_df = pd.read_csv(os.path.join(base_path, "unified_job_descriptions.csv"))

print("Resume Dataset Shape:", resume_df.shape)
print("JD Dataset Shape:", jd_df.shape)

print("\nResume Columns:")
print(resume_df.columns.tolist())

print("\nJob Description Columns:")
print(jd_df.columns.tolist())

Resume Dataset Shape: (17932, 5)
JD Dataset Shape: (3651, 5)

Resume Columns:
['resume_id', 'source', 'raw_text', 'skills', 'category']

Job Description Columns:
['job_title', 'description', 'source', 'required_skills', 'experience_level']


In [2]:
print("Resume Categories:")
print(resume_df["category"].value_counts().head(20))

print("\nJob Titles:")
print(jd_df["job_title"].value_counts().head(20))

Resume Categories:
category
HR Officer                                                                                 340
Project Coordinator (Civil)                                                                340
Site Engineer                                                                              340
Civil Engineer                                                                             340
Management Trainee - Mechanical                                                            339
Business Development Executive                                                             339
Mechanical Designer                                                                        339
AI Engineer                                                                                339
Database Administrator (DBA)                                                               339
Asst. Manager/ Manger (Administrative)                                                     339
System Administrator (

In [3]:
# Find resumes and JDs with similar role names

for i in range(min(10, len(resume_df))):
    print(
        "Resume:", i,
        "| Category:", resume_df.iloc[i]["category"]
    )

print("\n--- JOB DESCRIPTIONS ---")

for i in range(min(20, len(jd_df))):
    print(
        "JD:", i,
        "| Job Title:", jd_df.iloc[i]["job_title"]
    )

Resume: 0 | Category: Senior Software Engineer
Resume: 1 | Category: Machine Learning (ML) Engineer
Resume: 2 | Category: Executive/ Senior Executive- Trade Marketing, Hygiene Products
Resume: 3 | Category: Business Development Executive
Resume: 4 | Category: Senior iOS Engineer
Resume: 5 | Category: AI Engineer
Resume: 6 | Category: Senior iOS Engineer
Resume: 7 | Category: Senior iOS Engineer
Resume: 8 | Category: Mechanical Engineer
Resume: 9 | Category: Business Development Executive

--- JOB DESCRIPTIONS ---
JD: 0 | Job Title: Data Analyst
JD: 1 | Job Title: Data Reporting Analyst
JD: 2 | Job Title: Data Analyst (Power BI/Python)
JD: 3 | Job Title: Data & Reporting Analyst
JD: 4 | Job Title: Data Quality Analyst (Remote Opportunity)
JD: 5 | Job Title: Data Analyst
JD: 6 | Job Title: Data Reporting Analyst- (***YORK, PA***)
JD: 7 | Job Title: Product Data Analyst
JD: 8 | Job Title: Technical Data Analyst
JD: 9 | Job Title: Data Integration Analyst
JD: 10 | Job Title: Business Data 

In [4]:
# Show exact resume categories and JD titles
# so we can identify good and bad matching pairs

print("Resume Categories:")
print(resume_df["category"].value_counts().head(20))

print("\nJD Job Titles:")
print(jd_df["job_title"].value_counts().head(20))

Resume Categories:
category
HR Officer                                                                                 340
Project Coordinator (Civil)                                                                340
Site Engineer                                                                              340
Civil Engineer                                                                             340
Management Trainee - Mechanical                                                            339
Business Development Executive                                                             339
Mechanical Designer                                                                        339
AI Engineer                                                                                339
Database Administrator (DBA)                                                               339
Asst. Manager/ Manger (Administrative)                                                     339
System Administrator (

In [5]:
def normalize_role(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("/", " ")
    return set(text.split())


good_pairs = []
bad_pairs = []

# GOOD MATCHES
for r_idx in range(len(resume_df)):
    resume_role = normalize_role(resume_df.iloc[r_idx]["category"])

    for j_idx in range(len(jd_df)):
        jd_role = normalize_role(jd_df.iloc[j_idx]["job_title"])

        common_words = resume_role.intersection(jd_role)

        if len(common_words) >= 1:
            good_pairs.append((r_idx, j_idx))

            if len(good_pairs) == 2:
                break

    if len(good_pairs) == 2:
        break


# BAD MATCHES
for r_idx in range(len(resume_df)):
    resume_role = normalize_role(resume_df.iloc[r_idx]["category"])

    for j_idx in range(len(jd_df)):
        jd_role = normalize_role(jd_df.iloc[j_idx]["job_title"])

        common_words = resume_role.intersection(jd_role)

        if len(common_words) == 0:
            bad_pairs.append((r_idx, j_idx))

            if len(bad_pairs) == 3:
                break

    if len(bad_pairs) == 3:
        break


print("GOOD MATCHES")
for r, j in good_pairs:
    print(
        f"Resume {r}: {resume_df.iloc[r]['category']} "
        f"--> JD {j}: {jd_df.iloc[j]['job_title']}"
    )

print("\nBAD MATCHES")
for r, j in bad_pairs:
    print(
        f"Resume {r}: {resume_df.iloc[r]['category']} "
        f"--> JD {j}: {jd_df.iloc[j]['job_title']}"
    )

GOOD MATCHES
Resume 0: Senior Software Engineer --> JD 89: Data Analyst Senior
Resume 0: Senior Software Engineer --> JD 95: Senior Data Analyst

BAD MATCHES
Resume 0: Senior Software Engineer --> JD 0: Data Analyst
Resume 0: Senior Software Engineer --> JD 1: Data Reporting Analyst
Resume 0: Senior Software Engineer --> JD 2: Data Analyst (Power BI/Python)


In [9]:
# Calculate semantic similarity for the selected validation pairs

validation_pairs = good_pairs + bad_pairs

print("=" * 70)
print("RESUME-JD MATCHING VALIDATION")
print("=" * 70)

for i, (r_idx, j_idx) in enumerate(validation_pairs, start=1):

    resume_text = str(resume_df.iloc[r_idx]["raw_text"])
    jd_text = str(jd_df.iloc[j_idx]["description"])

    resume_embedding = model.encode(resume_text)
    jd_embedding = model.encode(jd_text)

    score = cosine_similarity(
        [resume_embedding],
        [jd_embedding]
    )[0][0] * 100

    if i <= len(good_pairs):
        pair_type = "GOOD MATCH"
    else:
        pair_type = "BAD MATCH"

    print(f"\nTest {i} - {pair_type}")
    print("-" * 50)
    print("Resume Category :", resume_df.iloc[r_idx]["category"])
    print("Job Title       :", jd_df.iloc[j_idx]["job_title"])
    print("Match Score     :", round(float(score), 2), "%")

RESUME-JD MATCHING VALIDATION

Test 1 - GOOD MATCH
--------------------------------------------------
Resume Category : Senior Software Engineer
Job Title       : Data Analyst Senior
Match Score     : 36.58 %

Test 2 - GOOD MATCH
--------------------------------------------------
Resume Category : Senior Software Engineer
Job Title       : Senior Data Analyst
Match Score     : 40.31 %

Test 3 - BAD MATCH
--------------------------------------------------
Resume Category : Senior Software Engineer
Job Title       : Data Analyst
Match Score     : 51.46 %

Test 4 - BAD MATCH
--------------------------------------------------
Resume Category : Senior Software Engineer
Job Title       : Data Reporting Analyst
Match Score     : 32.94 %

Test 5 - BAD MATCH
--------------------------------------------------
Resume Category : Senior Software Engineer
Job Title       : Data Analyst (Power BI/Python)
Match Score     : 55.04 %


In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [10]:
# Find exact or strong role matches

def clean_role(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("/", " ")
    text = text.replace("(", " ")
    text = text.replace(")", " ")
    return set(text.split())


print("POTENTIAL GOOD MATCHES")
print("=" * 70)

count = 0

for r_idx in range(len(resume_df)):
    resume_role = clean_role(resume_df.iloc[r_idx]["category"])

    for j_idx in range(len(jd_df)):
        jd_role = clean_role(jd_df.iloc[j_idx]["job_title"])

        # Remove generic seniority words
        resume_role_clean = resume_role - {"senior", "junior", "lead", "sr"}
        jd_role_clean = jd_role - {"senior", "junior", "lead", "sr"}

        # Calculate overlap
        common = resume_role_clean.intersection(jd_role_clean)

        # Require at least 2 meaningful common words
        if len(common) >= 2:
            print(
                f"Resume {r_idx}: {resume_df.iloc[r_idx]['category']}"
                f"  -->  JD {j_idx}: {jd_df.iloc[j_idx]['job_title']}"
                f"  | Common: {common}"
            )

            count += 1

            if count >= 10:
                break

    if count >= 10:
        break

POTENTIAL GOOD MATCHES
Resume 0: Senior Software Engineer  -->  JD 530: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 534: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 535: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 561: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 562: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 570: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 605: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 619: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 637: Software Engineer  | Common: {'engineer', 'software'}
Resume 0: Senior Software Engineer  -->  JD 648: S

In [11]:
# Select validated good and bad pairs

good_pairs = [
    (0, 530),
    (0, 534)
]

bad_pairs = [
    (0, 0),
    (0, 1),
    (0, 2)
]

print("GOOD MATCH PAIRS")
print("=" * 70)

for r, j in good_pairs:
    print(
        f"Resume {r}: {resume_df.iloc[r]['category']}"
        f" --> JD {j}: {jd_df.iloc[j]['job_title']}"
    )

print("\nBAD MATCH PAIRS")
print("=" * 70)

for r, j in bad_pairs:
    print(
        f"Resume {r}: {resume_df.iloc[r]['category']}"
        f" --> JD {j}: {jd_df.iloc[j]['job_title']}"
    )

GOOD MATCH PAIRS
Resume 0: Senior Software Engineer --> JD 530: Software Engineer
Resume 0: Senior Software Engineer --> JD 534: Software Engineer

BAD MATCH PAIRS
Resume 0: Senior Software Engineer --> JD 0: Data Analyst
Resume 0: Senior Software Engineer --> JD 1: Data Reporting Analyst
Resume 0: Senior Software Engineer --> JD 2: Data Analyst (Power BI/Python)


In [12]:
# Show different resume categories so we can select different resumes

for i in range(20):
    print(
        f"Resume {i}: {resume_df.iloc[i]['category']}"
    )

Resume 0: Senior Software Engineer
Resume 1: Machine Learning (ML) Engineer
Resume 2: Executive/ Senior Executive- Trade Marketing, Hygiene Products
Resume 3: Business Development Executive
Resume 4: Senior iOS Engineer
Resume 5: AI Engineer
Resume 6: Senior iOS Engineer
Resume 7: Senior iOS Engineer
Resume 8: Mechanical Engineer
Resume 9: Business Development Executive
Resume 10: Mechanical Designer
Resume 11: Asst. Manager/ Manger (Administrative)
Resume 12: Machine Learning (ML) Engineer
Resume 13: Mechanical Designer
Resume 14: Mechanical Engineer
Resume 15: Database Administrator (DBA)
Resume 16: System Administrator (Operation & Maintenance of Server, Storage & Service Desk System)
Resume 17: Executive/ Senior Executive- Trade Marketing, Hygiene Products
Resume 18: Project Coordinator (Civil)
Resume 19: Executive/ Sr. Executive -IT


In [13]:
# Show first occurrence of different resume categories

seen_categories = set()
selected_resumes = []

for i in range(len(resume_df)):
    category = str(resume_df.iloc[i]["category"]).strip()

    if category not in seen_categories:
        seen_categories.add(category)
        selected_resumes.append(i)

    if len(selected_resumes) >= 10:
        break

for i in selected_resumes:
    print(
        f"Resume {i}: {resume_df.iloc[i]['category']}"
    )

Resume 0: Senior Software Engineer
Resume 1: Machine Learning (ML) Engineer
Resume 2: Executive/ Senior Executive- Trade Marketing, Hygiene Products
Resume 3: Business Development Executive
Resume 4: Senior iOS Engineer
Resume 5: AI Engineer
Resume 8: Mechanical Engineer
Resume 10: Mechanical Designer
Resume 11: Asst. Manager/ Manger (Administrative)
Resume 15: Database Administrator (DBA)


In [14]:
# Final validation pairs
# 2 clearly good matches + 3 clearly bad matches

good_pairs = [
    (0, 530),   # Senior Software Engineer -> Software Engineer
    (1, 530)    # ML Engineer -> Software Engineer
]

bad_pairs = [
    (2, 530),   # Trade Marketing -> Software Engineer
    (3, 530),   # Business Development -> Software Engineer
    (11, 530)   # Administrative Manager -> Software Engineer
]

validation_pairs = good_pairs + bad_pairs

print("FINAL VALIDATION PAIRS")
print("=" * 70)

for i, (r, j) in enumerate(validation_pairs, start=1):

    if i <= len(good_pairs):
        pair_type = "GOOD MATCH"
    else:
        pair_type = "BAD MATCH"

    print(
        f"Test {i} - {pair_type}\n"
        f"Resume {r}: {resume_df.iloc[r]['category']}\n"
        f"JD {j}: {jd_df.iloc[j]['job_title']}\n"
    )

FINAL VALIDATION PAIRS
Test 1 - GOOD MATCH
Resume 0: Senior Software Engineer
JD 530: Software Engineer

Test 2 - GOOD MATCH
Resume 1: Machine Learning (ML) Engineer
JD 530: Software Engineer

Test 3 - BAD MATCH
Resume 2: Executive/ Senior Executive- Trade Marketing, Hygiene Products
JD 530: Software Engineer

Test 4 - BAD MATCH
Resume 3: Business Development Executive
JD 530: Software Engineer

Test 5 - BAD MATCH
Resume 11: Asst. Manager/ Manger (Administrative)
JD 530: Software Engineer



In [16]:
# Calculate semantic similarity for all 5 validation pairs
import numpy as np

print("=" * 70)
print("FINAL RESUME-JD MATCHING VALIDATION")
print("=" * 70)

results = []

for i, (r_idx, j_idx) in enumerate(validation_pairs, start=1):

    resume_text = str(resume_df.iloc[r_idx]["raw_text"])
    jd_text = str(jd_df.iloc[j_idx]["description"])

    resume_embedding = model.encode(resume_text)
    jd_embedding = model.encode(jd_text)

    score = cosine_similarity(
        [resume_embedding],
        [jd_embedding]
    )[0][0] * 100

    if i <= len(good_pairs):
        pair_type = "GOOD MATCH"
    else:
        pair_type = "BAD MATCH"

    results.append({
        "Test": i,
        "Type": pair_type,
        "Resume": resume_df.iloc[r_idx]["category"],
        "Job": jd_df.iloc[j_idx]["job_title"],
        "Score": round(float(score), 2)
    })

    print(f"\nTest {i} - {pair_type}")
    print("-" * 50)
    print("Resume :", resume_df.iloc[r_idx]["category"])
    print("JD     :", jd_df.iloc[j_idx]["job_title"])
    print("Score  :", round(float(score), 2), "%")


# Summary
good_scores = [x["Score"] for x in results if x["Type"] == "GOOD MATCH"]
bad_scores = [x["Score"] for x in results if x["Type"] == "BAD MATCH"]

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

print("Good Match Scores:", good_scores)
print("Bad Match Scores :", bad_scores)

print("Average Good Match Score:",
      round(np.mean(good_scores), 2), "%")

print("Average Bad Match Score:",
      round(np.mean(bad_scores), 2), "%")

FINAL RESUME-JD MATCHING VALIDATION

Test 1 - GOOD MATCH
--------------------------------------------------
Resume : Senior Software Engineer
JD     : Software Engineer
Score  : 45.76 %

Test 2 - GOOD MATCH
--------------------------------------------------
Resume : Machine Learning (ML) Engineer
JD     : Software Engineer
Score  : 37.6 %

Test 3 - BAD MATCH
--------------------------------------------------
Resume : Executive/ Senior Executive- Trade Marketing, Hygiene Products
JD     : Software Engineer
Score  : 14.03 %

Test 4 - BAD MATCH
--------------------------------------------------
Resume : Business Development Executive
JD     : Software Engineer
Score  : 29.78 %

Test 5 - BAD MATCH
--------------------------------------------------
Resume : Asst. Manager/ Manger (Administrative)
JD     : Software Engineer
Score  : 21.48 %

VALIDATION SUMMARY
Good Match Scores: [45.76, 37.6]
Bad Match Scores : [14.03, 29.78, 21.48]
Average Good Match Score: 41.68 %
Average Bad Match Score: 2

In [4]:
# First resume and first job description

resume = resume_df.iloc[0]["raw_text"]
job_description = jd_df.iloc[0]["description"]

print("Resume Preview:\n")
print(resume[:500])   # First 500 characters

print("\n" + "="*80 + "\n")

print("Job Description Preview:\n")
print(job_description[:500])   # First 500 characters

Resume Preview:

Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud infrastructure services and am well acquainted with them. Currently in search of role that offers more of development. Technical Support
Troubleshooting
Collaboration
Documentation
System Monitoring
Software Deployment
Training & Mentorship
Industry Trends
Field Visits







Job Description Preview:

job overview were seeking a data analyst to turn reports from dispatch, safety, accounting, and maintenance into actionable insights and visual dashboards. youll help drive smarter decisions across our trucking operations by translating data into strategies that improve efficiency and performance. key responsibilities analyze data across departments to uncover trends and recommend improvements build and maintain dashboards to track kpis cost per mile, fuel efficiency, driver performance, etc. co


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Create embeddings
resume_embedding = model.encode(resume)
jd_embedding = model.encode(job_description)

# Calculate similarity
similarity = cosine_similarity(
    [resume_embedding],
    [jd_embedding]
)

match_score = similarity[0][0] * 100

print(f"Resume-JD Match Score: {match_score:.2f}%")

Resume-JD Match Score: 51.46%


In [6]:
# Get skills
resume_skills = str(resume_df.iloc[0]["skills"]).split(",")
jd_skills = str(jd_df.iloc[0]["required_skills"]).split(",")

# Clean spaces
resume_skills = [skill.strip().lower() for skill in resume_skills]
jd_skills = [skill.strip().lower() for skill in jd_skills]

# Find matched and missing skills
matched_skills = list(set(resume_skills) & set(jd_skills))
missing_skills = list(set(jd_skills) - set(resume_skills))

print("Matched Skills:")
print(matched_skills)

print("\nMissing Skills:")
print(missing_skills)

Matched Skills:
[]

Missing Skills:
['nan']


In [7]:
print("Resume Skills:")
print(resume_df["skills"].head(5))

print("\nJD Required Skills:")
print(jd_df["required_skills"].head(5))

Resume Skills:
0    ['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...
1    ['Data Analysis', 'Data Analytics', 'Business ...
2    ['Software Development', 'Machine Learning', '...
3    ['accounts payables', 'accounts receivables', ...
4    ['Analytical reasoning', 'Compliance testing k...
Name: skills, dtype: object

JD Required Skills:
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: required_skills, dtype: object


In [8]:
from textwrap import fill

print(fill(jd_df.iloc[0]["description"], width=100))

job overview were seeking a data analyst to turn reports from dispatch, safety, accounting, and
maintenance into actionable insights and visual dashboards. youll help drive smarter decisions
across our trucking operations by translating data into strategies that improve efficiency and
performance. key responsibilities analyze data across departments to uncover trends and recommend
improvements build and maintain dashboards to track kpis cost per mile, fuel efficiency, driver
performance, etc. collaborate with team leads to identify reporting needs and deliver insights
conduct etl processes to clean and prepare data present findings clearly via reports and
visualizations requirements proficient in excel and power bi, tableau or similar tools strong data
analysis and critical thinking skills experience with sql and etl processes preferred bonus
familiarity with python or r excellent communication skillsable to simplify complex data for
stakeholders understanding of logistics or trucking 

In [10]:
# List of common technical skills
skills_list = [
    "Python", "Java", "C++", "SQL", "Machine Learning",
    "Deep Learning", "TensorFlow", "PyTorch", "AWS",
    "Docker", "Kubernetes", "Linux", "Git", "Power BI",
    "Excel", "Hadoop", "Spark", "Hive", "Tableau",
    "NLP", "Flask", "Django", "MongoDB", "MySQL",
    "ETL", "R"
]

# Get first job description
jd_text = jd_df.iloc[0]["description"].lower()

# Extract skills mentioned in JD
jd_skills = []

for skill in skills_list:
    if skill.lower() in jd_text:
        jd_skills.append(skill)

print("Skills Found in Job Description:")
print(jd_skills)

Skills Found in Job Description:
['Python', 'SQL', 'Power BI', 'Excel', 'Tableau', 'ETL', 'R']


In [11]:
import ast

# Resume skills ko list me convert karo
resume_skills = ast.literal_eval(resume_df.iloc[0]["skills"])

# Lowercase for comparison
resume_skills_lower = [skill.lower() for skill in resume_skills]
jd_skills_lower = [skill.lower() for skill in jd_skills]

# Matched skills
matched_skills = [
    skill for skill in jd_skills
    if skill.lower() in resume_skills_lower
]

# Missing skills
missing_skills = [
    skill for skill in jd_skills
    if skill.lower() not in resume_skills_lower
]

print("Resume Skills:")
print(resume_skills)

print("\nJD Skills:")
print(jd_skills)

print("\nMatched Skills:")
print(matched_skills)

print("\nMissing Skills:")
print(missing_skills)

Resume Skills:
['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapreduce', 'Spark', 'Java', 'Machine Learning', 'Cloud', 'Hdfs', 'YARN', 'Core Java', 'Data Science', 'C++', 'Data Structures', 'DBMS', 'RDBMS', 'Informatica', 'Talend', 'Amazon Redshift', 'Microsoft Azure']

JD Skills:
['Python', 'SQL', 'Power BI', 'Excel', 'Tableau', 'ETL', 'R']

Matched Skills:
['Python']

Missing Skills:
['SQL', 'Power BI', 'Excel', 'Tableau', 'ETL', 'R']


In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ast

model = SentenceTransformer("all-MiniLM-L6-v2")

skills_list = [
    "Python", "Java", "C++", "SQL", "Machine Learning",
    "Deep Learning", "TensorFlow", "PyTorch", "AWS",
    "Docker", "Kubernetes", "Linux", "Git", "Power BI",
    "Excel", "Hadoop", "Spark", "Hive", "Tableau",
    "NLP", "Flask", "Django", "MongoDB", "MySQL",
    "ETL", "R"
]

def match_resume_to_jd(resume_index, jd_index):

    resume_text = resume_df.iloc[resume_index]["raw_text"]
    jd_text = jd_df.iloc[jd_index]["description"]

    # Similarity Score
    resume_embedding = model.encode(resume_text)
    jd_embedding = model.encode(jd_text)

    score = cosine_similarity(
        [resume_embedding],
        [jd_embedding]
    )[0][0] * 100

    # Resume Skills
    resume_skills = ast.literal_eval(resume_df.iloc[resume_index]["skills"])
    resume_lower = [s.lower() for s in resume_skills]

    # JD Skills
    jd_found = []

    for skill in skills_list:
        if skill.lower() in jd_text.lower():
            jd_found.append(skill)

    matched = [s for s in jd_found if s.lower() in resume_lower]
    missing = [s for s in jd_found if s.lower() not in resume_lower]

    return {
        "Resume ID": resume_df.iloc[resume_index]["resume_id"],
        "Job Title": jd_df.iloc[jd_index]["job_title"],
        "Match Score": round(score,2),
        "Matched Skills": matched,
        "Missing Skills": missing
    }

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD h

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while request

In [13]:
result = match_resume_to_jd(0,0)

result

{'Resume ID': '0_r1',
 'Job Title': 'Data Analyst',
 'Match Score': np.float32(51.46),
 'Matched Skills': ['Python'],
 'Missing Skills': ['SQL', 'Power BI', 'Excel', 'Tableau', 'ETL', 'R']}

In [14]:
results = []

resume_index = 0   # First Resume

for jd_index in range(len(jd_df)):

    result = match_resume_to_jd(resume_index, jd_index)

    results.append(result)

print("Total Matches:", len(results))

Total Matches: 3651


In [15]:
results = sorted(
    results,
    key=lambda x: x["Match Score"],
    reverse=True
)

top5 = results[:5]

top5

[{'Resume ID': '0_r1',
  'Job Title': 'Machine Learning',
  'Match Score': np.float32(68.07),
  'Matched Skills': ['Python', 'Machine Learning'],
  'Missing Skills': ['Deep Learning',
   'TensorFlow',
   'PyTorch',
   'Docker',
   'Kubernetes',
   'Git',
   'R']},
 {'Resume ID': '0_r1',
  'Job Title': 'Machine Learning',
  'Match Score': np.float32(67.55),
  'Matched Skills': ['Python', 'Machine Learning', 'Spark'],
  'Missing Skills': ['SQL', 'AWS', 'Docker', 'Git', 'R']},
 {'Resume ID': '0_r1',
  'Job Title': 'Machine Learning',
  'Match Score': np.float32(66.43),
  'Matched Skills': ['Python', 'Java', 'Hadoop', 'Spark', 'Hive'],
  'Missing Skills': ['SQL',
   'AWS',
   'Power BI',
   'Excel',
   'Tableau',
   'MongoDB',
   'MySQL',
   'R']},
 {'Resume ID': '0_r1',
  'Job Title': 'Machine Learning',
  'Match Score': np.float32(65.39),
  'Matched Skills': ['Machine Learning'],
  'Missing Skills': ['R']},
 {'Resume ID': '0_r1',
  'Job Title': 'Sr. Data Analyst',
  'Match Score': np.flo

In [16]:
for i, job in enumerate(top5, start=1):

    print("="*60)

    print(f"Rank {i}")

    print("Job Title :", job["Job Title"])

    print("Match Score :", job["Match Score"])

    print("Matched Skills :", job["Matched Skills"])

    print("Missing Skills :", job["Missing Skills"])

    print("="*60)

Rank 1
Job Title : Machine Learning
Match Score : 68.07
Matched Skills : ['Python', 'Machine Learning']
Missing Skills : ['Deep Learning', 'TensorFlow', 'PyTorch', 'Docker', 'Kubernetes', 'Git', 'R']
Rank 2
Job Title : Machine Learning
Match Score : 67.55
Matched Skills : ['Python', 'Machine Learning', 'Spark']
Missing Skills : ['SQL', 'AWS', 'Docker', 'Git', 'R']
Rank 3
Job Title : Machine Learning
Match Score : 66.43
Matched Skills : ['Python', 'Java', 'Hadoop', 'Spark', 'Hive']
Missing Skills : ['SQL', 'AWS', 'Power BI', 'Excel', 'Tableau', 'MongoDB', 'MySQL', 'R']
Rank 4
Job Title : Machine Learning
Match Score : 65.39
Matched Skills : ['Machine Learning']
Missing Skills : ['R']
Rank 5
Job Title : Sr. Data Analyst
Match Score : 64.62
Matched Skills : ['Python']
Missing Skills : ['SQL', 'Git', 'Excel', 'Tableau', 'R']


In [17]:
results_df = pd.DataFrame(results)

results_df.head()

,Resume ID,Job Title,Match Score,Matched Skills,Missing Skills
0,0_r1,Machine Learning,68.070000,"[Python, Machine Learning]","[Deep Learning, TensorFlow, PyTorch, Docker, K..."
1,0_r1,Machine Learning,67.550003,"[Python, Machine Learning, Spark]","[SQL, AWS, Docker, Git, R]"
2,0_r1,Machine Learning,66.430000,"[Python, Java, Hadoop, Spark, Hive]","[SQL, AWS, Power BI, Excel, Tableau, MongoDB, ..."
3,0_r1,Machine Learning,65.389999,[Machine Learning],[R]
4,0_r1,Sr. Data Analyst,64.620003,[Python],"[SQL, Git, Excel, Tableau, R]"


In [18]:
results_df.to_csv("resume_jd_matching_results.csv", index=False)

print("Results saved successfully!")

Results saved successfully!


In [19]:
best_match = results_df.sort_values(
    by="Match Score",
    ascending=False
).iloc[0]

print(best_match)

Resume ID                                                      0_r1
Job Title                                          Machine Learning
Match Score                                                   68.07
Matched Skills                           [Python, Machine Learning]
Missing Skills    [Deep Learning, TensorFlow, PyTorch, Docker, K...
Name: 0, dtype: object


In [1]:
print(resume_df["category"].value_counts().head(20))

NameError: name 'resume_df' is not defined